In [0]:
# https://docs.databricks.com/aws/en/mlflow/

In [0]:
%pip install shap lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/275.7 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 9.6 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=c3a188dcecb55b28efbb3abd22ab718a9558e6d56e6f56a40107b68066095e98
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Notebook: 03_Model_Training
import mlflow
import mlflow.sklearn
from databricks import feature_store

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt # Needed for saving SHAP plots

# Import SHAP and LIME libraries
import shap
import lime
import lime.lime_tabular


In [0]:
# --- Parameters ---
# These could be passed as widgets or job parameters
n_estimators = 150
max_depth = 10
random_state = 42

mlflow.autolog(
    log_input_examples=True,
    log_model_signatures=True, # Keep signature logging if desired
    log_models=False,  # IMPORTANT: Disable autologging models for fs.log_model
    silent=True
)

In [0]:
# --- Load Data ---
# Assume 'prepared_df' (with target) and 'fs_table_name' are available or passed
# Option 1: Re-run previous steps if needed (not ideal for jobs)
# Option 2: Load base data and use Feature Store client to join features

# Load base data again (or from Delta Lake) - need primary key and target
# This assumes 01_Data_Ingestion was run or data is accessible
try:
    # Using the path from your example code
    base_df = spark.read.format("delta").load("/mnt/adventureworks/prepared_data2")
    base_df = base_df.select("primary_key", "TotalDue") # Need target variable and key
    print(f"Loaded base data for training. Count: {base_df.count()}")
except Exception as e:
    print(f"Could not load base data: {e}")
    dbutils.notebook.exit("Failed to load base data for training.")

Loaded base data for training. Count: 31465


In [0]:
# Using the feature table name from your example code
fs_table_name = "databricks_us.adventureworks_db.sales_order_features2"

fs = feature_store.FeatureStoreClient()

# Create Training Set by joining base data (target) with features from Feature Store
# Create a DataFrame with primary keys and timestamps (use OrderDate or current time)
# For simplicity, using just the primary key from our base_df
lookup_keys_df = base_df.select("primary_key")
print("Lookup keys head:")
lookup_keys_df.show(5) # Check keys

Lookup keys head:
+-----------+
|primary_key|
+-----------+
|      43659|
|      43660|
|      43661|
|      43662|
|      43663|
+-----------+
only showing top 5 rows


In [0]:

print("Base data head:")
display(base_df.limit(5)) # Check base data

Base data head:


primary_key,TotalDue
43659,23153.234
43660,1457.3289
43661,36865.8
43662,32474.932
43663,472.3108


In [0]:
try:
    training_set = fs.create_training_set(
        df=base_df, # DataFrame containing labels and primary keys
        feature_lookups=[
            feature_store.FeatureLookup(
                table_name=fs_table_name,
                lookup_key="primary_key"
            )
        ],
        label="TotalDue",
        exclude_columns=["primary_key"] # Exclude non-feature/non-label columns
    )
    training_pd = training_set.load_df().toPandas() # Load data into Pandas for scikit-learn
    print("Training set created successfully.")
    print(f"Training data shape: {training_pd.shape}")

except Exception as e:
    print(f"Error creating training set from Feature Store: {e}")
    dbutils.notebook.exit("Feature Store training set creation failed.")


Training set created successfully.
Training data shape: (31465, 8)


In [0]:
artifact_path="model",

In [0]:
%python
# --- Train/Test Split ---
# Ensure 'TotalDue' exists before dropping
if 'TotalDue' not in training_pd.columns:
     raise ValueError("Target variable 'TotalDue' not found in the training set DataFrame.")

X = training_pd.drop("TotalDue", axis=1)
y = training_pd["TotalDue"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

# --- Model Training, Logging, and Explainability ---
with mlflow.start_run() as run:
    # Log parameters (Autolog might capture some, but explicit is good too)
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("random_state", random_state)
    mlflow.log_param("feature_table", fs_table_name)

    # Train the model
    print("Training RandomForestRegressor...")
    rf = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, random_state=random_state)
    rf.fit(X_train, y_train)
    print("Training complete.")

    # Make predictions
    y_pred_train = rf.predict(X_train) # For signature inference if needed later
    y_pred_test = rf.predict(X_test)

    # Log metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    r2 = r2_score(y_test, y_pred_test)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    print(f"Metrics logged: RMSE={rmse}, R2={r2}")

    # Infer model signature using test set prediction for better representation
    from mlflow.models.signature import infer_signature
    # Use X_train and corresponding predictions for signature
    signature = infer_signature(X_train, y_pred_train)
    print("Model signature inferred.")

    # Log the model using the Feature Store API for better integration
    print("Logging model with Feature Store context...")
    fs.log_model(
        model=rf,
        artifact_path="model", # Name within MLflow run artifacts
        flavor=mlflow.sklearn,
        training_set=training_set, # Pass the training_set object
        signature=signature, # Include the inferred signature
        registered_model_name=None # Register in the next step
    )
    print("Model logged successfully using fs.log_model.")


X_train shape: (25172, 7), X_test shape: (6293, 7)
Training RandomForestRegressor...
Training complete.
Metrics logged: RMSE=414.7132572134326, R2=0.9989973901733561
Model signature inferred.
Logging model with Feature Store context...


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:406: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Uploading artifacts:   0%|          | 0/14 [00:00<?, ?it/s]

2025/04/09 19:03:45 INFO mlflow.tracking._tracking_service.client: 🏃 View run overjoyed-ox-875 at: adb-2035045055759449.9.azuredatabricks.net/ml/experiments/d474d03e6dee4105b3a608e7ac6227fb/runs/7582449a41b14a19ad3499d2d599d837.
2025/04/09 19:03:45 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: adb-2035045055759449.9.azuredatabricks.net/ml/experiments/d474d03e6dee4105b3a608e7ac6227fb.


Model logged successfully using fs.log_model.


In [0]:
# ========================================
# ===== SHAP Analysis ===================
# ========================================
print("Starting SHAP analysis...")
# Use TreeExplainer for efficiency with RandomForest
explainer = shap.TreeExplainer(rf)
# Calculate SHAP values for the test set (can use a subset for large data)
shap_values = explainer.shap_values(X_test)
print("SHAP values calculated.")

# --- Log SHAP Summary Plot ---
print("Generating and logging SHAP summary plot...")
shap_summary_plot_path = "/tmp/shap_summary_plot.png"
# Create the plot - use show=False to prevent direct display here
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.savefig(shap_summary_plot_path, bbox_inches='tight') # Save the plot
plt.close() # Close plot to prevent double display
mlflow.log_artifact(shap_summary_plot_path, "shap_plots") # Log as artifact
print("SHAP summary plot logged.")

# --- Log SHAP Dependence Plots (Example for top 2 features) ---
# Get global feature importance from mean absolute SHAP values
vals = np.abs(shap_values).mean(0)
feature_importance = pd.DataFrame(list(zip(X_train.columns, vals)), columns=['col_name','feature_importance_vals'])
feature_importance.sort_values(by=['feature_importance_vals'], ascending=False, inplace=True)
top_features = feature_importance['col_name'].head(2).tolist() # Get top 2 feature names

for feature in top_features:
    print(f"Generating and logging SHAP dependence plot for: {feature}")
    shap_dependence_plot_path = f"/tmp/shap_dependence_plot_{feature}.png"
    shap.dependence_plot(feature, shap_values, X_test, show=False)
    plt.savefig(shap_dependence_plot_path, bbox_inches='tight')
    plt.close()
    mlflow.log_artifact(shap_dependence_plot_path, "shap_plots")
print("SHAP dependence plots logged.")

Starting SHAP analysis...
SHAP values calculated.
Generating and logging SHAP summary plot...
SHAP summary plot logged.
Generating and logging SHAP dependence plot for: SubTotal
Generating and logging SHAP dependence plot for: TaxAmt
SHAP dependence plots logged.


In [0]:
# enable autologging
mlflow.sklearn.autolog()

In [0]:
# --- Model Training with MLflow ---
with mlflow.start_run() as run:
    # Train the model
    rf = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, random_state=random_state)
    rf.fit(X_train, y_train)

    # Make predictions
    y_pred = rf.predict(X_test)

    print("Model logged with Feature Store context.")

2025/04/08 12:01:19 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:406: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2025/04/08 12:01:30 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.12/site-packages/ml

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

2025/04/08 12:01:34 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:406: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2025/04/08 12:01:34 INFO mlflow.tracking._tracking_service.client: 🏃 View run wistful-wren-907 at: adb-2035045055759449.9.azuredatabricks.net/ml/experime

Model logged with Feature Store context.
